# Phase 2: Legal NER: MLA decoder

This is the **Legal NER** part of the full `mla` notebook (`notebooks/CS781_mla_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Fine Tuning

In [7]:
import collections

def strict_scores(gold, pred, labels):
    """
    gold, pred: {doc_id: [[start, end, label, (text)], ...]}
    Exact match on (doc_id, start, end, label). Macro over `labels`.
    Returns dict with macro_p / macro_r / macro_f1, per-label dict, and micro_f1.
    """
    def to_set(d):
        out = set()
        for did, ents in d.items():
            for e in ents:
                out.add((str(did), int(e[0]), int(e[1]), e[2]))
        return out

    G, P = to_set(gold), to_set(pred)
    tp, gp, pp = collections.Counter(), collections.Counter(), collections.Counter()
    for x in G: gp[x[3]] += 1
    for x in P: pp[x[3]] += 1
    for x in G & P: tp[x[3]] += 1

    per = {}
    for lab in labels:
        p = tp[lab] / pp[lab] if pp[lab] else 0.0
        r = tp[lab] / gp[lab] if gp[lab] else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        per[lab] = {"p": p, "r": r, "f1": f, "support": gp[lab], "pred": pp[lab], "tp": tp[lab]}

    n = len(labels)
    macro_p = sum(v["p"] for v in per.values()) / n
    macro_r = sum(v["r"] for v in per.values()) / n
    macro_f1 = sum(v["f1"] for v in per.values()) / n
    TP, PP, GG = sum(tp.values()), sum(pp.values()), sum(gp.values())
    mp, mr = (TP / PP if PP else 0.0), (TP / GG if GG else 0.0)
    micro_f1 = 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0
    return {"macro_p": macro_p, "macro_r": macro_r, "macro_f1": macro_f1,
            "micro_f1": micro_f1, "per_label": per}

def print_scores(res):
    print(f"MACRO  P={res['macro_p']:.4f}  R={res['macro_r']:.4f}  F1={res['macro_f1']:.4f}  | micro F1={res['micro_f1']:.4f}")
    print(f"{'label':10s} {'P':>7s} {'R':>7s} {'F1':>7s} {'gold':>6s} {'pred':>6s} {'tp':>6s}")
    for lab, v in sorted(res["per_label"].items(), key=lambda kv: -kv[1]["support"]):
        print(f"{lab:10s} {v['p']:7.3f} {v['r']:7.3f} {v['f1']:7.3f} {v['support']:6d} {v['pred']:6d} {v['tp']:6d}")

# ---------- self-test on toy data ----------
_L = ["JUDGE", "COURT", "DATE"]
_gold = {"d1": [[0, 5, "JUDGE", "Smith"], [10, 20, "COURT", "High Court"]],
         "d2": [[3, 9, "DATE", "1 May"]]}

# perfect predictions (with and without optional text field)
_r = strict_scores(_gold, {"d1": [[0, 5, "JUDGE"], [10, 20, "COURT", "x"]], "d2": [[3, 9, "DATE"]]}, _L)
assert abs(_r["macro_f1"] - 1.0) < 1e-9, _r

# boundary off by one and wrong label must both count as wrong
_r = strict_scores(_gold, {"d1": [[0, 6, "JUDGE"], [10, 20, "JUDGE"]], "d2": [[3, 9, "DATE"]]}, _L)
assert _r["per_label"]["JUDGE"]["tp"] == 0 and _r["per_label"]["COURT"]["tp"] == 0
assert _r["per_label"]["DATE"]["f1"] == 1.0
assert abs(_r["macro_f1"] - 1/3) < 1e-9, _r["macro_f1"]

# empty predictions -> zeros; duplicate predictions must not inflate counts
assert strict_scores(_gold, {}, _L)["macro_f1"] == 0.0
_r = strict_scores(_gold, {"d2": [[3, 9, "DATE"], [3, 9, "DATE"]]}, _L)
assert _r["per_label"]["DATE"]["pred"] == 1 and _r["per_label"]["DATE"]["p"] == 1.0
print("✅ strict scorer self-test passed")

✅ strict scorer self-test passed


In [7]:
import os, json, math, time, random, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

for name in ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "strict_scores", "print_scores"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cell that defines it."

# ============================================================
# 0. Config
# ============================================================
WINDOW, STRIDE = 1024, 768            # tokens per window / step (256 overlap)
VAL_FRACTION, SEED = 0.15, 42
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0

P2_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive not mounted — saving Phase-2 files locally (lost if the runtime disconnects).")
    P2_DIR = "CS781_Capstone_Project_Phase_2"
os.makedirs(P2_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if use_bf16 else torch.float16
PAD_ID = tokenizer.token_to_id("[PAD]")
PAD_ID = 1 if PAD_ID is None else PAD_ID
def amp(): return torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE)

# ============================================================
# 1. Load Phase-2 files
# ============================================================
BASE = "Phase_2_LNER"
def _path(fn): return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"{BASE}/{fn}", repo_type="dataset", token=HF_TOKEN)
def _json(fn):
    with open(_path(fn), "r", encoding="utf-8") as f: return json.load(f)

text1, gold1 = _json("Train/fold1-text.json"), _json("Train/fold1-gold.json")
text2, gold2 = _json("Train/fold2-text.json"), _json("Train/fold2-gold.json")
text3 = _json("Test/fold3-text.json")
with open(_path("Train/labels.txt"), "r", encoding="utf-8") as f:
    LABELS = [l.split()[0] for l in f if l.strip()]
print(f"Labels ({len(LABELS)}): {LABELS}")
assert len(LABELS) == 12, "labels.txt did not parse into 12 labels — paste me the first lines of the file."

def as_text(v): return v.get("text") if isinstance(v, dict) else v

def parse_entity(e):
    if isinstance(e, (list, tuple)):
        return int(e[0]), int(e[1]), e[2], (e[3] if len(e) > 3 else None)
    if isinstance(e, dict):
        g = lambda *ks: next((e[k] for k in ks if k in e), None)
        return (int(g("start", "start_char", "begin")), int(g("end", "end_char")),
                g("label", "type", "entity"), g("text", "entity_text", "span"))
    raise ValueError(f"Unknown annotation format: {e!r}")

def doc_entities(gold, doc_id):
    v = gold.get(doc_id, [])
    if isinstance(v, dict):
        v = next((x for x in v.values() if isinstance(x, list)), [])
    return [parse_entity(e) for e in v]

# ---- offset convention: is the gold `end` exclusive (python slice) or inclusive? ----
excl = incl = 0
for tx, gd in [(text1, gold1), (text2, gold2)]:
    for did, v in tx.items():
        t = as_text(v)
        for s, e, lab, et in doc_entities(gd, did):
            if et is None: continue
            if t[s:e] == et: excl += 1
            elif t[s:e + 1] == et: incl += 1
print(f"Offset test -> text[s:e]==entity: {excl} | text[s:e+1]==entity: {incl}")
if excl == 0 and incl == 0:
    print("⚠️ Could not verify the convention (no entity-text field matched). Assuming END-EXCLUSIVE.")
END_EXCL = excl >= incl
print("Gold end offset is", "EXCLUSIVE" if END_EXCL else "INCLUSIVE")

# ============================================================
# 2. BIO label space + token labelling / span decoding
# ============================================================
L2I = {"O": 0}
for lab in LABELS:
    L2I["B-" + lab] = len(L2I); L2I["I-" + lab] = len(L2I)
I2L = {v: k for k, v in L2I.items()}
NL = len(L2I)
print(f"Classes: {NL} (O + B/I for each of {len(LABELS)} labels)")

def label_tokens(offsets, ents):
    """ents: [(start, end_exclusive, label)] -> BIO ids per token. Overlapping entities: first (longest) wins."""
    n = len(offsets)
    starts = np.array([o[0] for o in offsets]); ends = np.array([o[1] for o in offsets])
    y = [0] * n; conflicts = 0
    for s, e, lab in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        i = int(np.searchsorted(ends, s, side="right"))   # first token ending after s
        first = True
        while i < n and starts[i] < e:
            if ends[i] > starts[i]:
                if y[i] != 0: conflicts += 1
                else:
                    y[i] = L2I[("B-" if first else "I-") + lab]; first = False
            i += 1
    return y, conflicts

def decode_spans(text, offsets, tags):
    """Token tag ids -> [(start, end_exclusive, label)], whitespace trimmed."""
    spans, cur = [], None
    for i, t in enumerate(tags):
        a, b = offsets[i]
        if b <= a: continue
        name = I2L[int(t)]
        if name == "O":
            if cur: spans.append(cur); cur = None
            continue
        pre, lab = name.split("-", 1)
        if pre == "B" or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [a, b, lab]
        else:
            cur[1] = b
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out

def spans_out(text, offsets, tags):
    return [[int(s), int(e if END_EXCL else e - 1), lab, text[s:e]] for s, e, lab in decode_spans(text, offsets, tags)]

# ============================================================
# 3. Build docs, split by DOCUMENT, windows
# ============================================================
def build_docs(text_d, gold_d, prefix=""):
    docs = []
    for did, v in text_d.items():
        t = as_text(v); enc = tokenizer.encode(t)
        d = {"id": prefix + str(did), "text": t, "ids": enc.ids, "offsets": enc.offsets}
        if gold_d is not None:
            raw = doc_entities(gold_d, did)
            bad = {lab for _, _, lab, _ in raw if lab not in LABELS}
            assert not bad, f"Gold labels not in labels.txt: {bad}"
            d["gold_raw"] = [[s, e, lab] for s, e, lab, _ in raw]
            d["labels"], d["conflicts"] = label_tokens(
                d["offsets"], [(s, e if END_EXCL else e + 1, lab) for s, e, lab, _ in raw])
        docs.append(d)
    return docs

t0 = time.time()
all_docs = build_docs(text1, gold1, "f1:") + build_docs(text2, gold2, "f2:")
test_docs = build_docs(text3, None)
print(f"Tokenized {len(all_docs)} train docs + {len(test_docs)} test docs in {time.time()-t0:.0f}s")

order = list(range(len(all_docs))); random.Random(SEED).shuffle(order)
n_val = max(1, int(VAL_FRACTION * len(all_docs)))
val_docs = [all_docs[i] for i in order[:n_val]]
train_docs = [all_docs[i] for i in order[n_val:]]

def make_windows(n, W=WINDOW, S=STRIDE):
    if n <= W: return [0]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n: starts.append(n - W)
    return starts

train_windows = [(d["ids"][s:s + WINDOW], d["labels"][s:s + WINDOW])
                 for d in train_docs for s in make_windows(len(d["ids"]))]

tok_tr = sum(len(d["ids"]) for d in train_docs)
print(f"Train docs: {len(train_docs)} ({tok_tr:,} tokens) -> {len(train_windows)} windows of ≤{WINDOW} "
      f"(~{sum(len(x) for x,_ in train_windows)/1e6:.2f}M tokens/epoch)")
print(f"Val docs: {len(val_docs)} | Test docs: {len(test_docs)} | "
      f"docs longer than WINDOW: {sum(len(d['ids'])>WINDOW for d in all_docs)} | max tokens: {max(len(d['ids']) for d in all_docs)}")
print(f"Entity-token conflicts (overlapping gold): {sum(d['conflicts'] for d in all_docs)}")
val_cnt = collections.Counter(l for d in val_docs for _, _, l in d["gold_raw"])
print("Val entity counts:", dict(val_cnt.most_common()))

# class weights (sqrt inverse frequency, clipped)
cnt = np.bincount(np.concatenate([np.array(d["labels"]) for d in train_docs]), minlength=NL).astype(np.float64)
w = np.ones(NL)
if USE_CLASS_WEIGHTS:
    for c in range(1, NL): w[c] = float(np.clip(math.sqrt(cnt[0] / max(cnt[c], 1.0)), 1.0, CLASS_W_CLIP))
CLASS_W = torch.tensor(w, dtype=torch.float32, device=device) if USE_CLASS_WEIGHTS else None
print(f"O tokens: {100*cnt[0]/cnt.sum():.1f}% | class weights (min/max entity): {w[1:].min():.1f}/{w[1:].max():.1f}")

def collate_windows(items):
    L = max(len(x) for x, _ in items)
    X = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    Y = torch.full((len(items), L), -100, dtype=torch.long)
    for i, (x, y) in enumerate(items):
        X[i, :len(x)] = torch.tensor(x); Y[i, :len(y)] = torch.tensor(y)
    return X, Y

# ============================================================
# 4. ORACLE round-trip: gold -> token labels -> spans -> strict score  (the ceiling)
# ============================================================
o_docs = train_docs + val_docs
o_gold = {d["id"]: d["gold_raw"] for d in o_docs}
o_pred = {d["id"]: spans_out(d["text"], d["offsets"], d["labels"]) for d in o_docs}
o_res = strict_scores(o_gold, o_pred, LABELS)
print("\n=== ORACLE (ceiling for any model with this tokenizer + labelling) ===")
print_scores(o_res)
_lost = []
for d in o_docs:
    got = {(s, e, l) for s, e, l, _ in o_pred[d["id"]]}
    for s, e, l in d["gold_raw"]:
        if (s, e, l) not in got and len(_lost) < 5: _lost.append((d["id"], s, e, l, d["text"][s:e if END_EXCL else e + 1]))
print("Examples of gold spans the round-trip cannot reproduce:", _lost)
assert o_res["macro_f1"] > 0.8, "Oracle ceiling too low — offsets/tokenizer alignment is off. Paste this output to me."

# ============================================================
# 5. NER wrapper around the GQA decoder (no change to the LM class)
# ============================================================
class NERModel(nn.Module):
    """Token classifier: GQA decoder backbone (causal) + linear head.
    shift=k: token i is classified from the hidden state at position i+k (k-token lookahead)."""
    def __init__(self, lm, d_model, n_labels, dropout=0.1, shift=0):
        super().__init__()
        self.lm = lm
        self.shift = shift
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids):
        lm = self.lm
        x = lm.token_embedding(input_ids)
        for layer in lm.layers:
            x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        out = self.head(self.drop(lm.norm(x)))
        k = self.shift
        if k > 0:   # position i now holds the prediction for token i (made from position i+k)
            out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

# ============================================================
# 6. Prediction (windows averaged) + test-file writer
# ============================================================
@torch.no_grad()
def predict_docs(ner, docs, batch=8):
    ner.eval(); out = {}
    for d in docs:
        n = len(d["ids"])
        probs = torch.zeros(n, NL, device=device)
        starts = make_windows(n)
        for i in range(0, len(starts), batch):
            chunk = starts[i:i + batch]
            seqs = [d["ids"][s:s + WINDOW] for s in chunk]
            L = max(len(q) for q in seqs)
            x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
            for j, q in enumerate(seqs): x[j, :len(q)] = torch.tensor(q)
            with amp(): lg = ner(x.to(device))
            p = F.softmax(lg.float(), dim=-1)
            k = ner.shift
            for j, (s, q) in enumerate(zip(chunk, seqs)):
                valid = len(q) if (s + len(q) >= n) else len(q) - k   # last k tokens of a window have no lookahead
                probs[s:s + valid] += p[j, :valid]
        out[d["id"]] = spans_out(d["text"], d["offsets"], probs.argmax(-1).tolist())
    return out

def write_test_predictions(ner, name):
    preds = predict_docs(ner, test_docs)
    path = f"{P2_DIR}/prediction_{name}.json"
    with open(path, "w", encoding="utf-8") as f: json.dump(preds, f, ensure_ascii=False)
    print(f"Saved {len(preds)} test docs / {sum(len(v) for v in preds.values())} entities -> {path}")
    return path

# ============================================================
# 7. Trainer (optionally with distillation from a teacher NER model)
# ============================================================
def train_ner(ner, name, teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
              backbone_lr=1e-4, head_lr=1e-3, batch_size=8, weight_decay=0.01,
              warmup_frac=0.06, debug=False):
    tw = train_windows[:batch_size * 10] if debug else train_windows
    vdocs = val_docs[:10] if debug else val_docs
    if debug: epochs = 1; name = name + "_debug"
    vgold = {d["id"]: d["gold_raw"] for d in vdocs}
    ner.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)

    groups = {}
    for n_, p in ner.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
         for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / batch_size); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(optimizer, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))

    n_train = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    print(f"[{name}] trainable params: {n_train/1e6:.2f}M | windows: {len(tw)} | steps/epoch: {spe} | "
          f"KD: {'alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None and alpha > 0 else 'off'}")
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    best_f1, best_res, bad, history = -1.0, None, 0, []
    best_path = f"{P2_DIR}/{name}_best.pt"; t0 = time.time()

    if teacher is not None: assert teacher.shift == ner.shift, "teacher and student must use the same shift"
    for ep in range(epochs):
        ner.train(); idx = list(range(len(tw))); random.shuffle(idx)
        loss_sum = torch.tensor(0.0, device=device); nb = 0
        bar = tqdm(range(0, len(idx), batch_size), desc=f"{name} ep {ep+1}/{epochs}")
        for i in bar:
            X, Y = collate_windows([tw[j] for j in idx[i:i + batch_size]])
            X, Y = X.to(device), Y.to(device)
            if ner.shift > 0: Y[:, -ner.shift:] = -100        # the last k columns have no lookahead
            with amp(): lg = ner(X)
            lg = lg.float()
            loss = F.cross_entropy(lg.reshape(-1, NL), Y.reshape(-1), weight=CLASS_W, ignore_index=-100)
            if teacher is not None and alpha > 0:
                with torch.no_grad(), amp(): tl = teacher(X)
                m = Y != -100
                kd = F.kl_div(F.log_softmax(lg[m] / T, -1), F.softmax(tl.float()[m] / T, -1),
                              reduction="batchmean") * (T * T)
                loss = (1 - alpha) * loss + alpha * kd
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ner.parameters(), 1.0)
            optimizer.step(); sched.step()
            loss_sum += loss.detach(); nb += 1
            if nb % 20 == 0: bar.set_postfix(loss=f"{(loss_sum / nb).item():.4f}")

        res = strict_scores(vgold, predict_docs(ner, vdocs), LABELS)
        history.append({"epoch": ep + 1, "train_loss": (loss_sum / max(1, nb)).item(), "macro_f1": res["macro_f1"]})
        print(f"epoch {ep+1}: train loss {history[-1]['train_loss']:.4f} | val macro P/R/F1 = "
              f"{res['macro_p']:.4f}/{res['macro_r']:.4f}/{res['macro_f1']:.4f} | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best_f1:
            best_f1, best_res, bad = res["macro_f1"], res, 0
            torch.save({"model": ner.state_dict(), "epoch": ep + 1, "val_macro_f1": best_f1}, best_path)
        else:
            bad += 1
            if bad >= patience: print(f"Early stop (no improvement for {patience} epochs)."); break

    ner.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)["model"])
    mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    print(f"✅ [{name}] best val macro-F1 {best_f1:.4f} | {(time.time()-t0)/60:.1f} min | peak mem {mem:.1f} GB | saved {best_path}")
    return {"res": best_res, "history": history, "best_path": best_path, "minutes": (time.time() - t0) / 60,
            "peak_gb": mem, "trainable_params": n_train}

print("\n✅ Phase-2 toolkit ready.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Labels (12): ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Offset test -> text[s:e]==entity: 8393 | text[s:e+1]==entity: 0
Gold end offset is EXCLUSIVE
Classes: 25 (O + B/I for each of 12 labels)
Tokenized 70 train docs + 35 test docs in 2s
Train docs: 60 (399,913 tokens) -> 530 windows of ≤1024 (~0.54M tokens/epoch)
Val docs: 10 | Test docs: 35 | docs longer than WINDOW: 70 | max tokens: 23710
Entity-token conflicts (overlapping gold): 0
Val entity counts: {'PREC': 212, 'STAT': 171, 'DATE': 157, 'AUTH': 150, 'COURT': 97, 'RESP': 89, 'CASENO': 75, 'APP': 66, 'JUDGE': 32, 'WIT': 23, 'R.COUNSEL': 22, 'A.COUNSEL': 20}
O tokens: 90.8% | class weights (min/max entity): 6.1/10.0

=== ORACLE (ceiling for any model with this tokenizer + labelling) ===
MACRO  P=0.9866  R=0.9866  F1=0.9866  | micro F1=0.9819
label            P       R      F1   gold   pred     tp
DATE         1.000   1.000   1.000   1575   1575   1575
STAT         0.

In [ ]:
import random, math, torch, torch.nn as nn, torch.nn.functional as F
for n in ["train_ner", "write_test_predictions", "NERModel", "train_docs", "NL", "tokenizer", "device"]:
    assert n in globals(), f"'{n}' missing, run setup + Phase-2 cell 1 first"
assert "CustomDecoderLM" in globals() and hasattr(CustomDecoderLM(32, 64, 1, 1, 128, 64).layers[0].attn, "kv_up_proj"), \
    "CustomDecoderLM is not the MLA class, restart the runtime and run the MLA definition cell"
try: del model
except NameError: pass

MLA_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt"
MLA_TARGETS = ("q_up_proj", "kv_down_proj", "kv_up_proj", "out_proj")   # MLA analogues of q/k/v/o

def build_mla_ner(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          max_seq_len=4096, qk_rope_head_dim=64, tie_weights=True)
    ck = torch.load(MLA_CKPT, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("model_state_dict", ck)) if isinstance(ck, dict) else ck
    sd = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    lm_.load_state_dict(sd)                       # strict: fails loudly on any mismatch
    return NERModel(lm_, d_model=768, n_labels=NL, dropout=dropout).to(device)

# ---- adapters (own implementation) ----
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x): return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    """W = ((alpha/r) B A + I) W0, acting on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_adapters(ner, kind, r, alpha, targets=MLA_TARGETS):
    for p in ner.parameters(): p.requires_grad_(False)
    cls = LoRALinear if kind == "lora" else LoRMAPlusLinear
    for layer in ner.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    for p in ner.head.parameters(): p.requires_grad_(True)

@torch.no_grad()
def check_merge(mod):
    W0 = mod.base.weight; x = torch.randn(4, W0.shape[1], device=W0.device); ref = mod(x)
    Wm = W0 + mod.scale * (mod.B @ mod.A) if isinstance(mod, LoRALinear) else W0 + (mod.scale * (mod.B @ mod.A)) @ W0
    return (ref - F.linear(x, Wm)).abs().max().item()

def table(rows, ref=""):
    print(f"\n{'method':8s} {'lr':>7s} {'trainable':>11s} {'bestF1':>7s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
    for k, lr, tp, f1, p, r_, mi, gb in rows:
        print(f"{k:8s} {lr:7.0e} {tp:11,d} {f1:7.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")
    if ref: print(ref)

def row(kind, lr, out):
    return (kind, lr, out["trainable_params"], max(h["macro_f1"] for h in out["history"]),
            out["res"]["macro_p"], out["res"]["macro_r"], out["minutes"], out["peak_gb"])

# quick check: checkpoint loads
t = build_mla_ner(); print("MLA params:", f"{sum(p.numel() for p in t.lm.parameters())/1e6:.2f}M"); del t

MLA params: 87.02M


In [ ]:
DEBUG_MODE = False     # True first, then False
random.seed(1); torch.manual_seed(1)
ner = build_mla_ner()
out = train_ner(ner, "ner_mla_full_ft", epochs=10, patience=3, backbone_lr=1e-4, head_lr=1e-3,
                batch_size=8, debug=DEBUG_MODE)
if not DEBUG_MODE: write_test_predictions(ner, "mla_full_ft")
table([row("full-ft", 1e-4, out)])
del ner; torch.cuda.empty_cache()

[ner_mla_full_ft] trainable params: 87.04M | windows: 530 | steps/epoch: 67 | KD: off


ner_mla_full_ft ep 1/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 1.5841 | val macro P/R/F1 = 0.0631/0.1293/0.0832 | 0.1 min


ner_mla_full_ft ep 2/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.4614 | val macro P/R/F1 = 0.0765/0.2111/0.1087 | 0.3 min


ner_mla_full_ft ep 3/10:   0%|          | 0/67 [00:00<?, ?it/s]

... (11 outputs left out)

ner_mla_full_ft ep 9/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 9: train loss 0.0051 | val macro P/R/F1 = 0.2095/0.2326/0.2134 | 1.4 min


ner_mla_full_ft ep 10/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 10: train loss 0.0026 | val macro P/R/F1 = 0.2133/0.2436/0.2197 | 1.5 min
✅ [ner_mla_full_ft] best val macro-F1 0.2197 | 1.5 min | peak mem 15.3 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_mla_full_ft_best.pt
Saved 35 test docs / 5177 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_mla_full_ft.json

method        lr   trainable  bestF1       P       R   min    GB
full-ft    1e-04  87,038,233  0.2197  0.2133  0.2436   1.5  15.3


In [ ]:
DEBUG_MODE = False
RES = []
for lr in [1e-3, 3e-3]:
    random.seed(1); torch.manual_seed(1)
    ner = build_mla_ner()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    ner.eval()
    with torch.no_grad(): before = ner(ids)
    inject_adapters(ner, "lora", 8, 16)
    with torch.no_grad(): d = (before - ner(ids)).abs().max().item()
    assert d < 1e-4, f"not identity at init ({d})"; print(f"[lora] identity OK ({d:.1e})")
    name = f"ner_mla_lora_lr{lr:g}"
    out = train_ner(ner, name, epochs=20, patience=5, backbone_lr=lr, head_lr=1e-3, batch_size=8, debug=DEBUG_MODE)
    print("merge check:", f"{check_merge(ner.lm.layers[0].attn.q_up_proj):.1e}")
    if not DEBUG_MODE: write_test_predictions(ner, f"mla_lora_lr{lr:g}")
    RES.append(row("lora", lr, out)); del ner; torch.cuda.empty_cache()
table(RES)

[lora] identity OK (0.0e+00)
[ner_mla_lora_lr0.001] trainable params: 0.50M | windows: 530 | steps/epoch: 67 | KD: off


ner_mla_lora_lr0.001 ep 1/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.0895 | val macro P/R/F1 = 0.0068/0.0258/0.0108 | 0.1 min


ner_mla_lora_lr0.001 ep 2/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 1.3107 | val macro P/R/F1 = 0.0100/0.0596/0.0166 | 0.3 min


ner_mla_lora_lr0.001 ep 3/20:   0%|          | 0/67 [00:00<?, ?it/s]

... (71 outputs left out)

ner_mla_lora_lr0.003 ep 19/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 19: train loss 0.0316 | val macro P/R/F1 = 0.1582/0.1938/0.1700 | 2.8 min


ner_mla_lora_lr0.003 ep 20/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 20: train loss 0.0254 | val macro P/R/F1 = 0.1630/0.2015/0.1748 | 2.9 min
✅ [ner_mla_lora_lr0.003] best val macro-F1 0.1748 | 2.9 min | peak mem 13.3 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_mla_lora_lr0.003_best.pt
merge check: 1.4e-06
Saved 35 test docs / 5650 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_mla_lora_lr0.003.json

method        lr   trainable  bestF1       P       R   min    GB
lora       1e-03     495,385  0.1419  0.1184  0.1829   2.9  13.3
lora       3e-03     495,385  0.1748  0.1630  0.2015   2.9  13.3


In [ ]:
DEBUG_MODE = False
RES = []
for lr in [1e-3, 3e-3]:
    random.seed(1); torch.manual_seed(1)
    ner = build_mla_ner()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    ner.eval()
    with torch.no_grad(): before = ner(ids)
    inject_adapters(ner, "lorma", 9, 18)
    with torch.no_grad(): d = (before - ner(ids)).abs().max().item()
    assert d < 1e-4, f"not identity at init ({d})"; print(f"[lorma+] identity OK ({d:.1e})")
    name = f"ner_mla_lorma_lr{lr:g}"
    out = train_ner(ner, name, epochs=20, patience=5, backbone_lr=lr, head_lr=1e-3, batch_size=8, debug=DEBUG_MODE)
    print("merge check:", f"{check_merge(ner.lm.layers[0].attn.q_up_proj):.1e}")
    if not DEBUG_MODE: write_test_predictions(ner, f"mla_lorma_lr{lr:g}")
    RES.append(row("lorma+", lr, out)); del ner; torch.cuda.empty_cache()
table(RES)

[lorma+] identity OK (0.0e+00)
[ner_mla_lorma_lr0.001] trainable params: 0.74M | windows: 530 | steps/epoch: 67 | KD: off


ner_mla_lorma_lr0.001 ep 1/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.0393 | val macro P/R/F1 = 0.0066/0.0258/0.0105 | 0.1 min


ner_mla_lorma_lr0.001 ep 2/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 1.2653 | val macro P/R/F1 = 0.0160/0.0767/0.0257 | 0.3 min


ner_mla_lorma_lr0.001 ep 3/20:   0%|          | 0/67 [00:00<?, ?it/s]

... (71 outputs left out)

ner_mla_lorma_lr0.003 ep 19/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 19: train loss 0.0599 | val macro P/R/F1 = 0.1163/0.1761/0.1394 | 2.9 min


ner_mla_lorma_lr0.003 ep 20/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 20: train loss 0.0463 | val macro P/R/F1 = 0.1128/0.1635/0.1328 | 3.0 min
✅ [ner_mla_lorma_lr0.003] best val macro-F1 0.1394 | 3.0 min | peak mem 13.6 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_mla_lorma_lr0.003_best.pt
merge check: 7.6e-06
Saved 35 test docs / 6387 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_mla_lorma_lr0.003.json

method        lr   trainable  bestF1       P       R   min    GB
lorma+     1e-03     744,985  0.1336  0.1121  0.1739   3.0  13.6
lorma+     3e-03     744,985  0.1394  0.1163  0.1761   3.0  13.6


In [ ]:
DEBUG_MODE = False
PREFIX_LEN = 32

class PrefixMLANER(NERModel):
    def __init__(self, lm, d_model, n_labels, init_ids, dropout=0.1):
        super().__init__(lm, d_model, n_labels, dropout)
        for p in self.parameters(): p.requires_grad_(False)
        lm.eval()
        with torch.no_grad():
            dev = next(lm.parameters()).device
            pkv = lm(torch.tensor([init_ids], device=dev))["past_key_values"]   # (c_kv [1,P,128], k_rope [1,H,P,64]) per layer
        self.pre_c = nn.ParameterList([nn.Parameter(c.detach().clone()) for c, _ in pkv])
        self.pre_r = nn.ParameterList([nn.Parameter(r.detach().clone()) for _, r in pkv])
        for p in self.head.parameters(): p.requires_grad_(True)
    def hidden(self, input_ids):
        lm = self.lm; B = input_ids.shape[0]
        x = lm.token_embedding(input_ids)
        for i, layer in enumerate(lm.layers):
            x, _ = layer(x, lm.cos, lm.sin, past_kv=(self.pre_c[i].expand(B, -1, -1), self.pre_r[i].expand(B, -1, -1, -1)))
        return lm.norm(x)
    def forward(self, input_ids): return self.head(self.drop(self.hidden(input_ids)))

@torch.no_grad()
def plain_hidden(lm, ids):
    x = lm.token_embedding(ids)
    for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
    return lm.norm(x)

RES = []
for lr in [1e-3, 5e-3]:
    random.seed(1); torch.manual_seed(1)
    base = build_mla_ner()
    init_ids = train_docs[0]["ids"][:PREFIX_LEN]
    ner = PrefixMLANER(base.lm, 768, NL, init_ids).to(device); ner.eval()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    with torch.no_grad():
        h1 = ner.hidden(ids)
        h2 = plain_hidden(ner.lm, torch.cat([torch.tensor([init_ids], device=device), ids], 1))[:, PREFIX_LEN:]
    d = (h1 - h2).abs().max().item(); assert d < 1e-3, f"prefix path mismatch ({d})"; print(f"[prefix] init check OK ({d:.1e})")
    name = f"ner_mla_prefix_lr{lr:g}"
    out = train_ner(ner, name, epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, weight_decay=0.0, debug=DEBUG_MODE)
    if not DEBUG_MODE: write_test_predictions(ner, f"mla_prefix_lr{lr:g}")
    RES.append(row("prefix", lr, out)); del ner, base; torch.cuda.empty_cache()
table(RES)

[prefix] init check OK (7.9e-06)
[ner_mla_prefix_lr0.001] trainable params: 0.31M | windows: 530 | steps/epoch: 67 | KD: off


ner_mla_prefix_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.2718 | val macro P/R/F1 = 0.0000/0.0000/0.0000 | 0.1 min


ner_mla_prefix_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 1.7006 | val macro P/R/F1 = 0.0079/0.0301/0.0125 | 0.3 min


ner_mla_prefix_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (53 outputs left out)

ner_mla_prefix_lr0.005 ep 17/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 17: train loss 0.9030 | val macro P/R/F1 = 0.0279/0.1000/0.0432 | 2.3 min


ner_mla_prefix_lr0.005 ep 18/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 18: train loss 0.8924 | val macro P/R/F1 = 0.0286/0.0988/0.0434 | 2.4 min
Early stop (no improvement for 6 epochs).
✅ [ner_mla_prefix_lr0.005] best val macro-F1 0.0473 | 2.4 min | peak mem 13.1 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_mla_prefix_lr0.005_best.pt
Saved 35 test docs / 17288 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_mla_prefix_lr0.005.json

method        lr   trainable  bestF1       P       R   min    GB
prefix     1e-03     305,945  0.0346  0.0258  0.0656   1.8  13.1
prefix     5e-03     305,945  0.0473  0.0319  0.0990   2.4  13.1
